In [1]:
import pandas as pd
import numpy as np

Iterates through each semester folder and read in all valid files. Then, column names are standardized and time column is converted to a proper datetime format. All files within each semester are also concatenated together.



In [2]:
import os
import pandas as pd

base_path = "Logs-206-201-Datashop"

datasets = {}
all_dfs = []

for semester in sorted(os.listdir(base_path)):
    semester_path = os.path.join(base_path, semester)

    if not os.path.isdir(semester_path):
        continue

    semester_frames = []
    for file in sorted(os.listdir(semester_path)):
        if file.endswith((".csv", ".tab", ".txt")):
            file_path = os.path.join(semester_path, file)
            df = pd.read_csv(file_path, sep="\t")

            # Standardize columns immediately
            df.columns = (
                df.columns
                    .str.strip()
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("(", "", regex=False)
                    .str.replace(")", "", regex=False)
            )

            # Convert time before concatenation
            df["time"] = pd.to_datetime(df["time"], errors="coerce")

            df["semester"] = semester
            df["source_file"] = file

            semester_frames.append(df)
            all_dfs.append(df)

    if semester_frames:
        datasets[semester] = pd.concat(semester_frames, ignore_index=True)


Uses individual files stored in all_df and concatenates all of them into a single unified dataset.



In [3]:
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df.shape

combined_df.columns


Index(['time', 'anon_student_id', 'action', 'problem_name', 'class',
       'level_chapter', 'level_subchapter', 'session_id', 'problem_view',
       'selection', 'input', 'feedback_text', 'feedback_classification',
       'cf_code', 'cf_week_no', 'cf_institution', 'semester', 'source_file'],
      dtype='str')

In [5]:
for col in combined_df.columns:
    print(col, combined_df[col].dtype)

time datetime64[us]
anon_student_id int64
action str
problem_name str
class int64
level_chapter str
level_subchapter str
session_id int64
problem_view int64
selection str
input object
feedback_text str
feedback_classification str
cf_code object
cf_week_no int64
cf_institution str
semester str
source_file str


In [6]:
combined_df["input"] = combined_df["input"].astype("string")
combined_df["cf_code"] = combined_df["cf_code"].astype("string")

In [7]:
combined_df.to_parquet("combined_raw.parquet")